# Panel App: Target Audit

A multi-stage application for auditing the targets within our region orthophoto.

- Upload region image
- Select samples of region
- Set parameters for CV search
    - Visualize parameters
- Run parameters on full orthophoto
    - View output
- Generate binary mask
- Audit detected targets on map
- Save targets to local geoJSON



In [14]:
import param
import panel as pn
import geopandas as gpd
import matplotlib.pyplot as plt
from io import BytesIO
import json

pn.extension('filedropper')


class TargetAuditApp(param.Parameterized):
    # Parameters for tracking uploaded files
    region_image = param.Parameter(default=None)
    region_geojson = param.Parameter(default=None)

    # FileDropper widgets

    # Accepted filetypes bug for this widget: https://github.com/holoviz/panel/issues/7153
    # accepted_filetypes=["allowed/geojson", ".geojson"],
    image_dropper = pn.widgets.FileDropper(height=100, max_file_size ="500MB")
    geojson_dropper = pn.widgets.FileDropper(height=100, max_file_size ="100MB")

    def __init__(self, **params):
        super().__init__(**params)

        # Link FileDropper outputs to parameters
        self.image_dropper.param.watch(self._update_region_image, "value")
        self.geojson_dropper.param.watch(self._update_region_geojson, "value")

    # Update methods for parameters
    def _update_region_image(self, event):
        if event.new:
            self.region_image = event.new[0]  # Use the first file uploaded

    def _update_region_geojson(self, event):
        if event.new:
            first_file_name = list(event.new.keys())[0] # Dict of file names:bytes
            file_bytes_string = event.new[first_file_name].decode("utf-8") # Bytes to string
            self.region_geojson = json.loads(file_bytes_string)  # String to JSON dict

    # A method to display the uploaded region image
    def view_image(self):
        if self.region_image:
            try:
                image_data = BytesIO(self.region_image["content"])
                fig, ax = plt.subplots(figsize=(8, 8))
                img = plt.imread(image_data)
                ax.imshow(img)
                ax.axis('off')
                return pn.pane.Matplotlib(fig)
            except Exception as e:
                return f"Error displaying image: {e}"
        else:
            return "No image uploaded."

    # A method to display the GeoJSON region outline
    def view_geojson(self):
        if self.region_geojson:
            try:
                return pn.pane.JSON(self.region_geojson, depth=2, name="Uploaded GeoJSON")
                return f"{self.region_geojson}"
                # region_contour_data = self.region_geojson.decode("utf-8")
                # return f"{region_contour_data}"
                geojson_data = BytesIO(self.region_geojson["content"])
                gdf = gpd.read_file(geojson_data)
                print(gdf)  # Print the GeoDataFrame to the console
                return f"GeoJSON loaded successfully. Number of features: {len(gdf)}"
            except Exception as e:
                return f"Error processing GeoJSON: {e}"
        else:
            return "No GeoJSON uploaded."

    # Panel layout combining file droppers and visualizations
    def panel(self):
        return pn.Column(
            pn.Row(
                pn.Column("**Drop Region Image Here**", self.image_dropper),
                pn.Column("**Drop GeoJSON Here**", self.geojson_dropper),
            ),
            pn.Row(
                pn.Column("**Uploaded Region Image**", self.view_image),
                pn.Column("**Uploaded GeoJSON Outline**", self.view_geojson),
            ),
        )


# Run the app
target_audit_app = TargetAuditApp()
target_audit_app.panel().servable()

target_audit_app.panel()

BokehModel(combine_events=True, render_bundle={'docs_json': {'98416794-bf51-4b3c-9d5b-1efd4c811181': {'version…

In [10]:
target_audit_app.geojson_dropper.value

{'region_contour.geojson': b'{\n"type": "FeatureCollection",\n"name": "region_contour",\n"crs": { "type": "name", "properties": { "name": "urn:ogc:def:crs:OGC:1.3:CRS84" } },\n"features": [\n{ "type": "Feature", "properties": { "FID": 0 }, "geometry": { "type": "Polygon", "coordinates": [ [ [ -103.60354195463583, 30.250675823114307 ], [ -103.601930614656922, 30.250577387216087 ], [ -103.601931042836057, 30.250506613331293 ], [ -103.601374425811585, 30.250486215850458 ], [ -103.601375263905226, 30.250386536845081 ], [ -103.600687423408758, 30.2503079129831 ], [ -103.600046090996585, 30.250472024314352 ], [ -103.599014011216312, 30.25041712589999 ], [ -103.598328805183527, 30.250153098375581 ], [ -103.597974191366674, 30.249451986072671 ], [ -103.599188559218405, 30.247583868146478 ], [ -103.599731763661111, 30.247374848878962 ], [ -103.600475185189509, 30.24713556814169 ], [ -103.60086517298295, 30.247123784423042 ], [ -103.604254099224448, 30.247370780715062 ], [ -103.604834203395328, 

In [13]:
target_audit_app.region_geojson
type(target_audit_app.region_geojson)

dict